# LOGIT-01 — Modèle fidèle à celui de la DREES

Reproduit le modèle publié par la DREES à l'identique : mêmes variables, même
méthode, mêmes métriques. Il sert de référence et de point de comparaison avec le
modèle ajusté de LOGIT-02.

## Ce que le modèle répond

Le pipeline dit « VALIDE » ou « DOUTEUX ». Le modèle dit « cette proposition a
94 % de chances d'être exacte ». La différence est opérationnelle : avec un
statut on reprend ou on ne reprend pas, avec une probabilité on choisit un seuil
selon le risque accepté.

## Les quatre variables

| Variable | Ce qu'elle dit |
|---|---|
| Cohérence SIREN | les 9 premiers chiffres du SIRET proposé correspondent au SIREN de la PM |
| Statut ANS | notre pipeline a classé la proposition en VALIDE_FORT |
| Accord entre jeux de données | notre chaîne et celle de la DREES proposent le même SIRET |
| SIRET initial | le SIRET proposé est celui déjà déclaré dans FINESS |

## La limite de ce modèle

La cohérence SIREN est calculée avec le **SIREN corrigé par la revue manuelle**.
Cette information n'existe que sur les structures examinées : elle est renseignée
9 079 fois sur 141 924 candidats.

Ce modèle ne peut donc pas être appliqué au-delà de l'échantillon. Il reste la
référence méthodologique ; c'est LOGIT-02 qui produit les probabilités
exploitables.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))

import numpy as np
import pandas as pd

from src import logit
from src.drees import (
    charger_propositions, propositions_ans, charger_echantillon,
    table_apprentissage, COL_COHERENCE_REVUE, COL_COHERENCE_ANS,
)
from src.display import afficher_tableau
from config.settings import DREES_DIR, LOGIT_DIR

LOGIT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 60)

PROPOSITIONS = DREES_DIR / "ege_siret_proposals.parquet"
ECHANTILLON = DREES_DIR / "ege_review_sample.xlsx"
SEUIL = 0.90

## 1. Données

In [2]:
print("Propositions :")
propositions = charger_propositions(PROPOSITIONS)

print("\nEchantillon de revue :")
echantillon = charger_echantillon(ECHANTILLON)

print("\nTable d'apprentissage :")
table = table_apprentissage(propositions, echantillon, "ege")

y = table["y"]
poids = table["calibrated_weight"]
groupes = table["EJ"]

Propositions :
   141,924 candidats, 102,926 structures
   origine : {'both': 63133, 'none': 35248, 'ANS': 29863, 'Adrien': 13680}

Echantillon de revue :
   916 structures examinees, 839 avec un identifiant retenu

Table d'apprentissage :
   92,996 structures avec une proposition de notre chaine
   698 structures examinees avec une proposition
   11 a fermer ecartees -> 687 exploitables
   proposition correcte : 441 / 687 (85.4 % pondere)


In [3]:
print("Couverture de la variable de coherence corrigee :")
total = len(propositions)
renseignee = propositions[COL_COHERENCE_REVUE].notna().sum()
print(f"   {renseignee:,} / {total:,} candidats ({renseignee / total * 100:.1f} %)")
print("\nSur la table d'apprentissage :")
print(f"   {table[COL_COHERENCE_REVUE].notna().sum():,} / {len(table):,}")

Couverture de la variable de coherence corrigee :
   9,079 / 141,924 candidats (6.4 %)

Sur la table d'apprentissage :
   678 / 687


## 2. Les variables

Chaque modalité devient une variable binaire. La modalité absente sert de
référence : les odds ratios se lisent par rapport à elle.

In [4]:
def construire_variables(d):
    coherence = d[COL_COHERENCE_REVUE]
    return pd.DataFrame({
        "Initial SIRET = yes":
            d["is_initial_siret"].fillna(False).astype(bool).astype(int),
        "ANS status = VALIDE_FORT":
            (d["statut_EGE_ANS"].fillna("") == "VALIDE_FORT").astype(int),
        "Datasets = total consistency":
            (d["datasets_consistency_EGE"] == "Total consistency").astype(int),
        "Datasets = partial consistency":
            (d["datasets_consistency_EGE"] == "Partial consistency").astype(int),
        "SIREN coherence = yes":
            (coherence.fillna(False) == True).astype(int),
        "SIREN coherence missing":
            coherence.isna().astype(int),
    }, index=d.index)

X = construire_variables(table)
print(X.sum().rename("structures concernees").to_string())

Initial SIRET = yes               277
ANS status = VALIDE_FORT          167
Datasets = total consistency      242
Datasets = partial consistency     38
SIREN coherence = yes             394
SIREN coherence missing             9


## 3. Estimation

In [5]:
modele = logit.ajuster(X, y, poids)
resultats = logit.odds_ratios(modele)
afficher_tableau(resultats, "Coefficients et odds ratios", max_lignes=10)

Variable,Coefficient,Odds ratio,IC bas,IC haut,p-value
const,-1.115000,0.330000,0.310000,0.340000,0.000000
Initial SIRET = yes,0.384000,1.470000,1.320000,1.630000,0.000000
ANS status = VALIDE_FORT,1.182000,3.260000,2.950000,3.600000,0.000000
Datasets = total consistency,0.843000,2.320000,2.090000,2.580000,0.000000
Datasets = partial consistency,-4.746000,0.010000,0.010000,0.010000,0.000000
SIREN coherence = yes,4.141000,62.890000,57.380000,68.920000,0.000000
SIREN coherence missing,1.960000,7.100000,5.820000,8.670000,0.000000


### Comment lire un odds ratio

Un odds ratio de 62,9 ne signifie pas « 62 fois plus de chances ». Il signifie que
la **cote** — le rapport entre la probabilité d'avoir raison et celle d'avoir
tort — est multipliée par 62,9. L'effet sur la probabilité dépend du point de
départ.

Au-dessus de 1 l'effet est positif, en dessous négatif. « Ajusté » signifie : à
valeurs constantes des autres variables.

In [6]:
reference = {"SIREN coherence = yes": 62.93, "ANS status = VALIDE_FORT": 3.26,
             "Datasets = total consistency": 2.32, "Initial SIRET = yes": 1.52}
obtenus = dict(zip(resultats["Variable"], resultats["Odds ratio"]))

comparaison = pd.DataFrame([
    {"Variable": v, "Obtenu": obtenus.get(v), "Publie DREES": r,
     "Ecart": round(abs(obtenus.get(v, 0) - r), 2)}
    for v, r in reference.items()])
afficher_tableau(comparaison, "Comparaison aux coefficients publies", max_lignes=6)

Variable,Obtenu,Publie DREES,Ecart
SIREN coherence = yes,62.890000,62.930000,0.040000
ANS status = VALIDE_FORT,3.260000,3.260000,0.000000
Datasets = total consistency,2.320000,2.320000,0.000000
Initial SIRET = yes,1.470000,1.520000,0.050000


## 4. Validation croisée

Évaluer un modèle sur les données qui l'ont entraîné donne toujours un résultat
trop beau. Chaque observation reçoit donc une prédiction issue d'un modèle qui ne
l'a jamais vue.

Les plis sont groupés par entité juridique : deux établissements d'une même PM
partagent SIREN, adresse et souvent nom. Les séparer entre apprentissage et
validation reviendrait à donner la réponse au modèle.

In [7]:
proba = logit.predire_hors_echantillon(X, y, poids, groupes, n_plis=5)
afficher_tableau(logit.metriques(y, proba, poids), "Metriques hors echantillon",
                 max_lignes=8)

Metrique,Valeur
Score de Brier,0.054500
Brier de reference (taux moyen),0.124700
Reduction du Brier,56.3 %
AUC,0.916400
Prediction moyenne,0.849000
Taux observe,0.854000
Observations,687


Le **score de Brier** est l'écart quadratique moyen entre probabilité annoncée et
réalité. Seul il ne dit rien : on le compare à celui d'un modèle qui prédirait
toujours le taux moyen. L'écart mesure l'apport des variables.

L'**AUC** mesure la capacité à ordonner. Les deux sont nécessaires : un modèle
peut parfaitement classer tout en annonçant des probabilités absurdes.

## 5. Calibration

Vérification décisive. Sans elle, un seuil de 0,90 ne veut rien dire : parmi les
cas annoncés à 0,90, on doit en trouver environ 90 % de corrects.

In [8]:
afficher_tableau(logit.calibration(y, proba, poids, n_classes=5),
                 "Calibration par classe de probabilite", max_lignes=8)

Classe,Probabilite min,Probabilite max,Prediction moyenne,Taux observe,Effectif
1,0.001000,0.265000,0.135000,0.278000,160
2,0.270000,0.571000,0.320000,0.171000,115
3,0.581000,0.965000,0.928000,0.900000,140
4,0.970000,0.983000,0.980000,0.992000,144
5,0.986000,0.999000,0.996000,0.976000,128


## 6. Choix du seuil

Monter le seuil augmente la précision et réduit le volume. La dernière colonne
mérite attention : parmi les propositions rejetées, une part est correcte. Le
modèle n'élimine pas des erreurs, il isole des certitudes.

In [9]:
seuils = logit.table_seuils(y, proba, poids)
afficher_tableau(seuils, "Volume et precision selon le seuil", max_lignes=12)

ligne = seuils[seuils["Seuil"] == SEUIL].iloc[0]
print(f"\nAu seuil de {SEUIL:.2f} : {ligne['Acceptes (%)']:.1f} % acceptes, "
      f"{ligne['Corrects parmi acceptes (%)']:.1f} % corrects, "
      f"{ligne['Erreur residuelle (%)']:.1f} % d'erreur")

Seuil,Acceptes (%),Corrects parmi acceptes (%),Erreur residuelle (%),Corrects parmi rejetes (%),Structures acceptees
0.500000,84.000000,97.000000,3.000000,24.300000,431
0.600000,83.800000,97.100000,2.900000,25.100000,407
0.700000,83.700000,97.100000,2.900000,25.300000,398
0.800000,82.300000,98.400000,1.600000,25.300000,390
0.850000,81.900000,98.400000,1.600000,26.600000,380
0.900000,81.900000,98.400000,1.600000,26.700000,379
0.920000,81.900000,98.400000,1.600000,26.700000,378
0.950000,78.900000,98.300000,1.700000,37.000000,371
0.970000,69.100000,98.400000,1.600000,56.400000,264



Au seuil de 0.90 : 81.9 % acceptes, 98.4 % corrects, 1.6 % d'erreur


## 7. Export

In [10]:
FICHIER = LOGIT_DIR / "EGE_logit_modele_fidele.xlsx"

sortie = table[["EGE", "EJ", "siret_proposal", "Siret_retenu", "y",
                "calibrated_weight"]].copy()
sortie["probabilite"] = proba.round(4)
sortie["accepte"] = proba >= SEUIL

with pd.ExcelWriter(FICHIER, engine="openpyxl") as writer:
    resultats.to_excel(writer, sheet_name="Coefficients", index=False)
    logit.metriques(y, proba, poids).to_excel(writer, sheet_name="Metriques", index=False)
    logit.calibration(y, proba, poids).to_excel(writer, sheet_name="Calibration", index=False)
    seuils.to_excel(writer, sheet_name="Seuils", index=False)
    sortie.to_excel(writer, sheet_name="Predictions", index=False)

print(f"Export : {FICHIER}")

Export : /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/results/model_logit/EGE_logit_modele_fidele.xlsx
